In [ ]:

from __future__ import annotations

import atexit
import csv
import io
import sys
from datetime import datetime
from pathlib import Path

_NOTEBOOK_LOG_DIR = Path("logs")
_NOTEBOOK_LOG_DIR.mkdir(parents=True, exist_ok=True)
SESSION = datetime.now().strftime('%Y%m%d_%H%M%S')
NOTEBOOK_LOG_FILE = _NOTEBOOK_LOG_DIR / f"notebook_{SESSION}.log"

NOTEBOOK_CSV_FILE = _NOTEBOOK_LOG_DIR / "notebook_outputs.csv"
print("Notebook log file:", str(NOTEBOOK_LOG_FILE))
print("Notebook csv file (append):", str(NOTEBOOK_CSV_FILE))

class _Tee(io.TextIOBase):
    def __init__(self, original: io.TextIOBase, *, log_path: Path, csv_path: Path, stream: str, session: str) -> None:
        self._original = original
        self._fh = open(log_path, "a", encoding="utf-8")
        self._csv_fh = open(csv_path, "a", encoding="utf-8", newline="")
        self._csv = csv.writer(self._csv_fh)
        self._stream = stream
        self._session = session
        self._buf = ""
        try:
            if self._csv_fh.tell() == 0:
                self._csv.writerow(["session", "timestamp", "stream", "message"])
        except Exception:
            pass

    def _write_csv_line(self, message: str) -> None:
        if not message.strip():
            return
        try:
            ts = datetime.now().isoformat(timespec="seconds")
            self._csv.writerow([self._session, ts, self._stream, message])
        except Exception:
            pass

    def write(self, s: str) -> int:
        n = self._original.write(s)
        try:
            self._fh.write(s)
        except Exception:
            pass
        if s:
            self._buf += s
            while "\n" in self._buf:
                line, self._buf = self._buf.split("\n", 1)
                self._write_csv_line(line)
        return n

    def flush(self) -> None:
        try:
            self._original.flush()
        except Exception:
            pass
        try:
            self._fh.flush()
        except Exception:
            pass
        if self._buf:
            self._write_csv_line(self._buf.rstrip("\n"))
            self._buf = ""
        try:
            self._csv_fh.flush()
        except Exception:
            pass

    def close(self) -> None:
        try:
            self.flush()
        finally:
            for fh in (self._fh, self._csv_fh):
                try:
                    fh.close()
                except Exception:
                    pass

def _install_notebook_logging(*, log_file: Path, csv_file: Path, session: str) -> None:
    if getattr(sys.stdout, "_is_notebook_tee", False):
        return
    stdout_tee = _Tee(sys.stdout, log_path=log_file, csv_path=csv_file, stream="stdout", session=session)
    stderr_tee = _Tee(sys.stderr, log_path=log_file, csv_path=csv_file, stream="stderr", session=session)
    setattr(stdout_tee, "_is_notebook_tee", True)
    setattr(stderr_tee, "_is_notebook_tee", True)
    sys.stdout = stdout_tee  # type: ignore[assignment]
    sys.stderr = stderr_tee  # type: ignore[assignment]
    atexit.register(lambda: getattr(sys.stdout, "close", lambda: None)())
    atexit.register(lambda: getattr(sys.stderr, "close", lambda: None)())

_install_notebook_logging(log_file=NOTEBOOK_LOG_FILE, csv_file=NOTEBOOK_CSV_FILE, session=SESSION)

In [ ]:
import importlib

from kxor_code.algorithms.base_alg_step import ProblemRecord
from kxor_code.problem_set_generation.kxor_instance import KXORInstance

import kxor_code.algorithms.key_extraction_step as kes

importlib.reload(kes)
KeyExtractionStep = kes.KeyExtractionStep

print("VALID_STAGE1_BACKENDS:", KeyExtractionStep.VALID_STAGE1_BACKENDS)

record_path = (
    "/Users/julian/CodingProjects/UniT3/QuarticSpeedupK-XOR/"
    "z_Daphne/data/final_problem_set/kxor_instance_n18_k2_m147_ell2_eps0.15_kappa0.875_rho0.8.npz"
 )

instance = KXORInstance.load(record_path)
record = ProblemRecord(problem_id="notebook_problem", instance=instance)

record.add_field("ell", 2)
record.add_field("threshold", 0.0)

print("record_path:", record_path)
print("instance n:", record.instance.n)

from kxor_code.algorithms.compute_kikuchi_step import ComputeKikuchiStep
from kxor_code.algorithms.classical_eigenvalues_step import ClassicalEigenvaluesStep

ComputeKikuchiStep().execute(record)
record.add_field("num_eigenvalues", 5)
ClassicalEigenvaluesStep().execute(record)

stage1_backend = "precomputed_eigenvector"

try:
    import pennylane as qml
    stage2_backend = None
    print("PennyLane found; using circuit eigensolver for stage 2.")
except ModuleNotFoundError:
    print("PennyLane not found; using classical eigensolver for stage 2.")
    stage2_backend = "classical_eigsh"

step = KeyExtractionStep(stage1_backend=stage1_backend, stage2_backend=stage2_backend, evaluate=True)
stats = step.execute(record)

print("failed?", stats.failed)
if stats.failed:
    print("error:", stats.additional_data.get("error"))
    print("available fields:", sorted(record.fields.keys()))
else:
    z_hat = record.get_field("z_hat")
    print("n =", record.instance.n, "len(z_hat) =", len(z_hat))
    print("z_hat:", z_hat)
    for k in ["eval_advantage", "eval_hamming_frac", "eval_correlation"]:
        if k in stats.additional_data:
            print(k, stats.additional_data[k])

KeyboardInterrupt: 

In [ ]:

from __future__ import annotations

import importlib
import io
import logging
import re
import sys
from pathlib import Path
from typing import Any, Dict

import numpy as np

from kxor_code.algorithms.base_alg_step import ProblemRecord
from kxor_code.problem_set_generation.kxor_instance import KXORInstance
import kxor_code.algorithms.key_extraction_step as kes

def _make_file_logger(*, log_file: Path) -> logging.Logger:
    """Create an isolated logger that writes only to `log_file`."""
    log_file.parent.mkdir(parents=True, exist_ok=True)
    logger = logging.getLogger(f"final_problem_set.{log_file.stem}")
    logger.setLevel(logging.INFO)
    logger.propagate = False
    for h in list(logger.handlers):
        logger.removeHandler(h)
    handler = logging.FileHandler(log_file, mode="w", encoding="utf-8")
    handler.setLevel(logging.INFO)
    handler.setFormatter(logging.Formatter("%(asctime)s - %(name)s - %(levelname)s - %(message)s"))
    logger.addHandler(handler)
    return logger

class _StreamToLogger(io.TextIOBase):
    """File-like object that forwards writes to a logger (line-buffered)."""
    def __init__(self, logger: logging.Logger, level: int) -> None:
        self._logger = logger
        self._level = level
        self._buf = ""
    def write(self, s: str) -> int:
        if not s:
            return 0
        self._buf += s
        while "\n" in self._buf:
            line, self._buf = self._buf.split("\n", 1)
            if line.strip():
                self._logger.log(self._level, "STDIO: %s", line)
        return len(s)
    def flush(self) -> None:
        if self._buf.strip():
            self._logger.log(self._level, "STDIO: %s", self._buf.rstrip("\n"))
        self._buf = ""

_ROOT_HANDLER_TAG = "_final_problem_set_root_handler"

def _attach_root_file_handler(*, log_file: Path, level: int = logging.INFO) -> logging.Handler:
    """Capture library/module logging (logging.getLogger(__name__)) into the per-instance log."""
    root = logging.getLogger()
    for h in list(root.handlers):
        if getattr(h, _ROOT_HANDLER_TAG, False):
            root.removeHandler(h)
            try:
                h.close()
            except Exception:
                pass
    handler = logging.FileHandler(log_file, mode="a", encoding="utf-8")
    handler.setLevel(level)
    handler.setFormatter(logging.Formatter("%(asctime)s - %(name)s - %(levelname)s - %(message)s"))
    setattr(handler, _ROOT_HANDLER_TAG, True)
    root.addHandler(handler)
    if root.level > level:
        root.setLevel(level)
    return handler

def _parse_params_from_stem(stem: str) -> Dict[str, Any]:
    """Parse parameters from filenames like:
    kxor_instance_n18_k2_m147_ell2_eps0.15_kappa0.875_rho0.8
    """
    out: Dict[str, Any] = {}
    parts = stem.split("_")
    for part in parts:
        if part.startswith("n") and part[1:].isdigit():
            out["n"] = int(part[1:])
        elif part.startswith("k") and part[1:].isdigit():
            out["k"] = int(part[1:])
        elif part.startswith("m") and part[1:].isdigit():
            out["m"] = int(part[1:])
        elif part.startswith("ell") and part[3:].isdigit():
            out["ell"] = int(part[3:])
        elif part.startswith("eps"):
            try:
                out["eps"] = float(part[3:])
            except Exception:
                pass
        elif part.startswith("kappa"):
            try:
                out["kappa"] = float(part[5:])
            except Exception:
                pass
        elif part.startswith("rho"):
            try:
                out["rho"] = float(part[3:])
            except Exception:
                pass
    if "eps" not in out:
        m = re.search(r"eps([0-9]*\.?[0-9]+)", stem)
        if m:
            out["eps"] = float(m.group(1))
    if "kappa" not in out:
        m = re.search(r"kappa([0-9]*\.?[0-9]+)", stem)
        if m:
            out["kappa"] = float(m.group(1))
    if "rho" not in out:
        m = re.search(r"rho([0-9]*\.?[0-9]+)", stem)
        if m:
            out["rho"] = float(m.group(1))
    return out

def iterate(
    record_path: str,
    *,
    ell: int = 2,
    threshold: float = 0.0,
    num_eigenvalues: int = 5,
    stage2_backend: str | None = "classical_eigsh",
    log_dir: str | Path = "logs/final_problem_set/extraction_via_quantum_circuit",
) -> Dict[str, Any]:
    """Run the full pipeline for ONE instance file and write logs to its own file.

    Logs include:
    - step/pipeline logging (we pass `logger=...` to steps)
    - module logging via a temporary root FileHandler
    - any `print(...)` from imported code via stdout/stderr redirection
    - the full recovered key `z_hat`
    """
    record_path = str(record_path)
    log_dir = Path(log_dir)
    stem = Path(record_path).stem
    log_file = log_dir / f"{stem}.log"
    logger = _make_file_logger(log_file=log_file)
    logger.info("Starting iteration for %s", record_path)

    np.set_printoptions(threshold=np.inf, linewidth=200)
    old_out, old_err = sys.stdout, sys.stderr
    sys.stdout = _StreamToLogger(logger, logging.INFO)  # type: ignore[assignment]
    sys.stderr = _StreamToLogger(logger, logging.ERROR)  # type: ignore[assignment]
    root_handler: logging.Handler | None = None
    try:
        root_handler = _attach_root_file_handler(log_file=log_file, level=logging.INFO)
        importlib.reload(kes)
        KeyExtractionStep = kes.KeyExtractionStep
        instance = KXORInstance.load(record_path)
        record = ProblemRecord(problem_id=stem, instance=instance)
        record.add_field("ell", int(ell))
        record.add_field("threshold", float(threshold))
        logger.info("Loaded instance (n=%d, m=%d, k=%d)", int(instance.n), int(instance.m), int(instance.k))
        from kxor_code.algorithms.compute_kikuchi_step import ComputeKikuchiStep
        from kxor_code.algorithms.classical_eigenvalues_step import ClassicalEigenvaluesStep
        ComputeKikuchiStep(logger=logger).execute(record)
        record.add_field("num_eigenvalues", int(num_eigenvalues))
        ClassicalEigenvaluesStep(logger=logger).execute(record)
        step = KeyExtractionStep(
            stage1_backend="precomputed_eigenvector",
            stage2_backend=stage2_backend,
            evaluate=True,
            logger=logger,
)
        stats = step.execute(record)
        parsed = _parse_params_from_stem(stem)
        out: Dict[str, Any] = {
            "path": record_path,
            "stem": stem,
            "log_file": str(log_file),
            
            "n": int(getattr(record.instance, "n", parsed.get("n", 0))),
            "k": int(getattr(record.instance, "k", parsed.get("k", 0))),
            "m": int(getattr(record.instance, "m", parsed.get("m", 0))),
            "ell": int(parsed.get("ell", ell)),
            "eps": parsed.get("eps"),
            "kappa": parsed.get("kappa"),
            "rho": parsed.get("rho"),
            "failed": bool(stats.failed),
        }
        if stats.failed:
            out["error"] = stats.additional_data.get("error")
            logger.error("FAILED: %s", out["error"])
            return out
        z_hat = record.get_field("z_hat")
        out["len_z_hat"] = int(len(z_hat))
        out["eval_advantage"] = stats.additional_data.get("eval_advantage")
        out["eval_hamming_frac"] = stats.additional_data.get("eval_hamming_frac")
        out["eval_correlation"] = stats.additional_data.get("eval_correlation")
        logger.info("SUCCESS: len(z_hat)=%d", out["len_z_hat"])
        logger.info(
            "Metrics: adv=%s, ham_frac=%s, corr=%s",
            out["eval_advantage"],
            out["eval_hamming_frac"],
            out["eval_correlation"],
        )
        logger.info("z_hat (full): %s", np.array2string(np.asarray(z_hat, dtype=int), separator=", "))
        return out
    except Exception as exc:
        logger.exception("UNCAUGHT ERROR")
        return {
            "path": record_path,
            "stem": stem,
            "log_file": str(log_file),
            "failed": True,
            "error": str(exc),
        }
    finally:
        try:
            sys.stdout.flush()
            sys.stderr.flush()
        except Exception:
            pass
        sys.stdout, sys.stderr = old_out, old_err
        if root_handler is not None:
            root = logging.getLogger()
            try:
                root.removeHandler(root_handler)
                root_handler.close()
            except Exception:
                pass

In [ ]:


from __future__ import annotations

import importlib
import io
import logging
import re
import sys
from pathlib import Path
from typing import Any, Dict, Iterable

import numpy as np

from kxor_code.algorithms.base_alg_step import ProblemRecord
from kxor_code.problem_set_generation.kxor_instance import KXORInstance
import kxor_code.algorithms.key_extraction_step as kes


def _normalize_allowed_n(allowed_n: int | Iterable[int] | None) -> set[int] | None:
    if allowed_n is None:
        return None
    if isinstance(allowed_n, int):
        return {int(allowed_n)}
    return {int(x) for x in allowed_n}


def iterate_quantum(
    record_path: str,
    *,
    ell: int = 2,
    threshold: float = 0.0,
    num_eigenvalues: int = 5,

    max_n: int = 16,
    allowed_n: int | Iterable[int] | None = None,
    log_dir: str | Path = "logs/final_problem_set_quantum_stage2",
    stage2_circuit_phase_qubits: int | None = None,
    stage2_circuit_iters: int = 1,
    stage2_circuit_neighborhood: int = 1,
) -> Dict[str, Any]:
    """Run pipeline on ONE instance file, using quantum stage-2 circuit (small n only).

    Returns a dict matching iterate(...) style, with extra fields:
    - stage2_backend: 'schmidhuber_stage2_circuit'
    - skipped: True if filtered out by allowed_n/max_n

    If PennyLane is not available, this returns failed=True with an informative error.
    """

    record_path = str(record_path)
    log_dir = Path(log_dir)
    stem = Path(record_path).stem
    log_file = log_dir / f"{stem}.log"

    logger = _make_file_logger(log_file=log_file)
    logger.info("Starting iterate_quantum for %s", record_path)

    np.set_printoptions(threshold=np.inf, linewidth=200)

    allowed = _normalize_allowed_n(allowed_n)

    old_out, old_err = sys.stdout, sys.stderr
    sys.stdout = _StreamToLogger(logger, logging.INFO)  # type: ignore[assignment]
    sys.stderr = _StreamToLogger(logger, logging.ERROR)  # type: ignore[assignment]
    root_handler: logging.Handler | None = None

    try:
        try:
            import pennylane as _qml 
        except ModuleNotFoundError as exc:
            return {
                "path": record_path,
                "stem": stem,
                "log_file": str(log_file),
                "failed": True,
                "error": "PennyLane not found; quantum stage-2 requires 'pennylane' in this kernel.",
            }

        root_handler = _attach_root_file_handler(log_file=log_file, level=logging.INFO)

        importlib.reload(kes)
        KeyExtractionStep = kes.KeyExtractionStep

        instance = KXORInstance.load(record_path)
        n = int(instance.n)

        if allowed is not None and n not in allowed:
            msg = f"SKIP: instance n={n} not in allowed_n={sorted(allowed)}"
            logger.info(msg)
            return {
                "path": record_path,
                "stem": stem,
                "log_file": str(log_file),
                "failed": False,
                "skipped": True,
                "skip_reason": msg,
                "n": n,
            }

        if int(max_n) is not None and n > int(max_n):
            msg = f"SKIP: instance n={n} exceeds max_n={int(max_n)} (quantum stage-2 safety limit)"
            logger.info(msg)
            return {
                "path": record_path,
                "stem": stem,
                "log_file": str(log_file),
                "failed": False,
                "skipped": True,
                "skip_reason": msg,
                "n": n,
            }

        record = ProblemRecord(problem_id=stem, instance=instance)
        record.add_field("ell", int(ell))
        record.add_field("threshold", float(threshold))

        logger.info("Loaded instance (n=%d, m=%d, k=%d)", int(instance.n), int(instance.m), int(instance.k))

        from kxor_code.algorithms.compute_kikuchi_step import ComputeKikuchiStep
        from kxor_code.algorithms.classical_eigenvalues_step import ClassicalEigenvaluesStep

        ComputeKikuchiStep(logger=logger).execute(record)
        record.add_field("num_eigenvalues", int(num_eigenvalues))
        ClassicalEigenvaluesStep(logger=logger).execute(record)

        step = KeyExtractionStep(
            stage1_backend="precomputed_eigenvector",
            stage2_backend="schmidhuber_stage2_circuit",
            stage2_circuit_phase_qubits=stage2_circuit_phase_qubits,
            stage2_circuit_iters=stage2_circuit_iters,
            stage2_circuit_neighborhood=stage2_circuit_neighborhood,
            evaluate=True,
            logger=logger,
        )

        stats = step.execute(record)

        parsed = _parse_params_from_stem(stem)
        out: Dict[str, Any] = {
            "path": record_path,
            "stem": stem,
            "log_file": str(log_file),
            "stage2_backend": "schmidhuber_stage2_circuit",
            "n": int(getattr(record.instance, "n", parsed.get("n", 0))),
            "k": int(getattr(record.instance, "k", parsed.get("k", 0))),
            "m": int(getattr(record.instance, "m", parsed.get("m", 0))),
            "ell": int(parsed.get("ell", ell)),
            "eps": parsed.get("eps"),
            "kappa": parsed.get("kappa"),
            "rho": parsed.get("rho"),
            "failed": bool(stats.failed),
        }

        if stats.failed:
            out["error"] = stats.additional_data.get("error")
            logger.error("FAILED: %s", out["error"])
            return out

        z_hat = record.get_field("z_hat")
        out["len_z_hat"] = int(len(z_hat))
        out["eval_advantage"] = stats.additional_data.get("eval_advantage")
        out["eval_hamming_frac"] = stats.additional_data.get("eval_hamming_frac")
        out["eval_correlation"] = stats.additional_data.get("eval_correlation")

        logger.info("SUCCESS: len(z_hat)=%d", out["len_z_hat"])
        logger.info(
            "Metrics: adv=%s, ham_frac=%s, corr=%s",
            out["eval_advantage"],
            out["eval_hamming_frac"],
            out["eval_correlation"],
        )
        logger.info("z_hat (full): %s", np.array2string(np.asarray(z_hat, dtype=int), separator=", "))

        return out

    except Exception as exc:
        logger.exception("UNCAUGHT ERROR")
        return {
            "path": record_path,
            "stem": stem,
            "log_file": str(log_file),
            "failed": True,
            "error": str(exc),
        }
    finally:
        try:
            sys.stdout.flush()
            sys.stderr.flush()
        except Exception:
            pass
        sys.stdout, sys.stderr = old_out, old_err
        if root_handler is not None:
            root = logging.getLogger()
            try:
                root.removeHandler(root_handler)
                root_handler.close()
            except Exception:
                pass


In [ ]:

from pathlib import Path

import pandas as pd

base = Path("/Users/julian/CodingProjects/UniT3/QuarticSpeedupK-XOR")
final_dir = base / "z_Daphne/data/final_problem_set"
paths = sorted(final_dir.glob("kxor_instance_*.npz"))
print("Found", len(paths), "files in", str(final_dir))

results = []
for p in paths:
    results.append(
        iterate(
            str(p),
            stage2_backend="classical_eigsh",
            log_dir=base / "logs/final_problem_set",
        )
    )

df = pd.DataFrame(results)

print(df.to_string(index=False))

summary_csv = base / "logs" / "summary_all_instances.csv"
summary_csv.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(summary_csv, index=False)
print("Wrote summary CSV:", str(summary_csv))

KeyboardInterrupt: 

In [ ]:
from pathlib import Path

import pandas as pd

base = Path("/Users/julian/CodingProjects/UniT3/QuarticSpeedupK-XOR")
final_dir = base / "z_Daphne/data/final_problem_set"

_n = 13

paths = sorted(final_dir.glob(f"kxor_instance_n{_n}_*.npz"))
print("Found", len(paths), f"n={_n} files in", str(final_dir))

results = [
    iterate_quantum(
        str(p),
        allowed_n=_n, 
        max_n=_n,      
        log_dir=base / "logs/final_problem_set_quantum_stage2",
    )
    for p in paths
]

df = pd.DataFrame(results)

print(df.to_string(index=False))

summary_csv = base / "logs" / "summary_all_instances_quantum.csv"
summary_csv.parent.mkdir(parents=True, exist_ok=True)
write_header = not summary_csv.exists()
df.to_csv(summary_csv, mode="a", header=write_header, index=False)
print("Appended summary CSV:", str(summary_csv))
